In [2]:
import numpy as np
import uproot
import mplhep as mh
import matplotlib.pyplot as plt

from pathlib import Path
from matplotlib.patches import Rectangle


RUN_ORDER = ["2022", "2023", "2024", "2025"]

RUN_LABELS = {
    "2022": r"$2022\ (34.7\ fb^{-1})$",
    "2023": r"$2023\ (27.9\ fb^{-1})$",
    "2024": r"$2024\ (52.4\ fb^{-1})$",
    "2025": r"$2025\ (xx.x\ fb^{-1})$",
}

REGION_STYLE = {
    "All": {
        "facecolors": ["#8fbdc4", "#5ab2bf", "#00aec9", "#58c4a3"],
        "edgecolors": ["#005F77", "#005F77", "#005F77", "#005F77"],
    },
    "Barrel": {
        "facecolors": ["#87c7a7", "#4ac285", "#02c260", "#8fd19e"],
        "edgecolors": ["#003b0f", "#003b0f", "#003b0f", "#003b0f"],
    },
    "Disk1,2,3": {
        "facecolors": ["#9498d4", "#555dd4", "#020dcc", "#7b68ee"],
        "edgecolors": ["#000775", "#000775", "#000775", "#000775"],
    },
    "Disk4": {
        "facecolors": ["#fab4a2", "#fa7655", "#FF3300", "#ffb347"],
        "edgecolors": ["#CC0000", "#CC0000", "#CC0000", "#CC0000"],
    },
}

HATCHES = ["\\\\", "//", "oo", "--"]

IRPC_EXCLUDED_ROLLS = {
    "RE+4_R1_CH15_A",
    "RE+4_R1_CH16_A",
    "RE+3_R1_CH15_A",
    "RE+3_R1_CH16_A",
}

EFF_BINS = 200
EFF_RANGE = (0.0, 100.0)
VISIBLE_X_RANGE = (70.0, 100.0)


def make_region_mask(region: str, roll_names: np.ndarray) -> np.ndarray:
    roll_names = np.asarray(roll_names, dtype=str)

    if region == "All":
        return np.ones(len(roll_names), dtype=bool)

    if region == "Barrel":
        return np.array([name.startswith("W") for name in roll_names], dtype=bool)

    if region == "Endcap":
        return np.array([name.startswith("RE") for name in roll_names], dtype=bool)

    if region == "Disk1,2,3":
        prefixes = ("RE+1", "RE+2", "RE+3", "RE-1", "RE-2", "RE-3")
        return np.array([name.startswith(prefixes) for name in roll_names], dtype=bool)

    if region == "Disk4":
        prefixes = ("RE+4", "RE-4")
        return np.array([name.startswith(prefixes) for name in roll_names], dtype=bool)

    return np.array([name.startswith(region) for name in roll_names], dtype=bool)


def get_region_style(region: str) -> dict[str, list[str]]:
    if region == "Endcap":
        return REGION_STYLE["Disk1,2,3"]
    if region in REGION_STYLE:
        return REGION_STYLE[region]
    if region.startswith("W"):
        return REGION_STYLE["Barrel"]
    if region.startswith(("RE+1", "RE+2", "RE+3", "RE-1", "RE-2", "RE-3")):
        return REGION_STYLE["Disk1,2,3"]
    if region.startswith(("RE+4", "RE-4")):
        return REGION_STYLE["Disk4"]
    return REGION_STYLE["All"]


def make_irpc_mask(roll_names: np.ndarray) -> np.ndarray:
    roll_names = np.asarray(roll_names, dtype=str)
    return np.array([name in IRPC_EXCLUDED_ROLLS for name in roll_names], dtype=bool)


def safe_mean(values: np.ndarray) -> float:
    if len(values) == 0:
        return float("nan")
    return float(np.mean(values))


def load_roll_hists(root_path: Path):
    with uproot.open(root_path) as f:
        total_by_roll = f["total_by_roll"].to_hist()
        passed_by_roll = f["passed_by_roll"].to_hist()
    return total_by_roll, passed_by_roll


def compute_roll_efficiency(root_path: Path, region: str) -> dict:
    total_by_roll, passed_by_roll = load_roll_hists(root_path)

    total = total_by_roll.values()
    passed = passed_by_roll.values()
    roll_names = np.array(total_by_roll.axes[0], dtype=str)

    region_mask = make_region_mask(region, roll_names)
    irpc_mask = make_irpc_mask(roll_names)
    mask = region_mask & ~irpc_mask

    total = total[mask]
    passed = passed[mask]

    eff = np.divide(
        passed,
        total,
        out=np.zeros_like(total, dtype=float),
        where=(total > 0),
    ) * 100.0

    eff_valid = eff[total != 0]

    n_rolls_region = len(total)
    n_rolls_excluded_zero_total = int(np.count_nonzero(total == 0))
    n_rolls_under70 = int(np.count_nonzero(eff_valid <= 70))
    mean_above70 = safe_mean(eff_valid[eff_valid > 70])

    return {
        "eff": eff_valid,
        "n_rolls_region": n_rolls_region,
        "n_rolls_excluded_zero_total": n_rolls_excluded_zero_total,
        "n_rolls_under70": n_rolls_under70,
        "mean_above70": mean_above70,
    }


def build_legend(ax, patches, results_by_run: dict[str, dict]) -> None:
    extra = Rectangle((0, 0), 0.1, 0.1, fc="w", fill=False, edgecolor="none", linewidth=0)

    legend_header = [
        "",
        "",
        r"$Mean\ (>70\ \%)$",
        r"$\%\ (\leq70\ \%)$",
    ]

    legend_rows = {}
    for run_year in RUN_ORDER:
        result = results_by_run[run_year]
        n_total = result["n_rolls_region"]
        n_under70 = result["n_rolls_under70"]
        frac_under70 = 100.0 * n_under70 / n_total if n_total > 0 else float("nan")

        legend_rows[run_year] = [
            "",
            RUN_LABELS[run_year],
            f"       {result['mean_above70']:.1f} %",
            f"     {frac_under70:.1f} %",
        ]

    legend_handles = []
    legend_values = []

    for idx in range(len(legend_header)):
        if idx == 0:
            legend_handles += [extra] + [patches[run_year] for run_year in RUN_ORDER]
        else:
            legend_handles += [extra] * (len(RUN_ORDER) + 1)

        legend_values += [legend_header[idx]] + [legend_rows[run_year][idx] for run_year in RUN_ORDER]

    ax.legend(
        legend_handles,
        legend_values,
        ncol=len(legend_header),
        columnspacing=-0.6,
        handletextpad=-0.3,
        handlelength=2.0,
        handleheight=1.6,
        alignment="center",
        loc=(0.03, 0.48),
        fontsize=18,
    )


def hist_eff_by_roll(
    input_paths: dict[str, Path],
    region: str,
    output_path: Path,
) -> list[np.ndarray]:
    style = get_region_style(region)

    if len(style["facecolors"]) < len(RUN_ORDER):
        raise ValueError(f"Not enough facecolors for {len(RUN_ORDER)} runs in region '{region}'.")
    if len(style["edgecolors"]) < len(RUN_ORDER):
        raise ValueError(f"Not enough edgecolors for {len(RUN_ORDER)} runs in region '{region}'.")
    if len(HATCHES) < len(RUN_ORDER):
        raise ValueError(f"Not enough HATCHES entries for {len(RUN_ORDER)} runs.")

    results_by_run = {}
    for run_year in RUN_ORDER:
        if run_year not in input_paths:
            raise ValueError(f"Missing input path for run year '{run_year}'.")
        results_by_run[run_year] = compute_roll_efficiency(input_paths[run_year], region)

    mh.style.use(mh.styles.CMS)
    fig, ax = plt.subplots(figsize=(12, 8))

    mh.cms.label(
        ax=ax,
        data=True,
        llabel="Work in Progress",
        loc=0,
        year="Run 3",
        com=13.6,
        fontsize=24,
    )

    ax.set_xlabel("Efficiency [%]", fontsize=24)
    ax.set_ylabel("Number of Rolls", fontsize=24)
    ax.set_xlim(*VISIBLE_X_RANGE)

    ax.annotate(
        "Tag-and-Probe method",
        (0.96, 0.90),
        xycoords="axes fraction",
        fontsize=24,
        horizontalalignment="right",
    )

    ax.annotate(
        f"RPC {region}",
        (0.04, 0.90),
        xycoords="axes fraction",
        fontsize=24,
        horizontalalignment="left",
    )

    counts_all = []
    patches = {}

    for idx, run_year in enumerate(RUN_ORDER):
        eff = results_by_run[run_year]["eff"]

        count, bins, patch = ax.hist(
            eff[eff > 0],
            bins=EFF_BINS,
            range=EFF_RANGE,
            facecolor=style["facecolors"][idx],
            edgecolor=style["edgecolors"][idx],
            hatch=HATCHES[idx],
            alpha=1.0,
            align="mid",
            density=False,
            linewidth=2.0,
            histtype="stepfilled",
        )
        counts_all.append(count)
        patches[run_year] = patch[0]

    if len(counts_all) > 0:
        max_count = np.max(np.concatenate(counts_all))
        ax.set_ylim(0, max_count * 1.2 if max_count > 0 else 1.0)

    build_legend(ax, patches, results_by_run)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

    return [results_by_run[run_year]["eff"] for run_year in RUN_ORDER]


if __name__ == "__main__":
    input_paths = {
        "2022": Path("/home/joshin/workspace-gate/store/hdfs-joshin/rpc/tnp-merged-year/Run2022.root"),
        "2023": Path("/home/joshin/workspace-gate/store/hdfs-joshin/rpc/tnp-merged-year/Run2023.root"),
        "2024": Path("/home/joshin/workspace-gate/store/hdfs-joshin/rpc/tnp-merged-year/Run2024.root"),
        "2025": Path("/home/joshin/workspace-gate/store/hdfs-joshin/rpc/tnp-merged-year/Run2025.root"),
    }

    workspace = Path("/home/joshin/workspace-gate/RPCTnP/CMSSW_16_1_0_pre1/src/RPCDPGAnalysis/NanoAODTnP/plots")

    regions = ["Barrel", "Endcap"]
    for region in regions:
        hist_eff_by_roll(
            input_paths=input_paths,
            region=region,
            output_path=workspace / f"tnp_eff_hist_run3_{region}_with2025.png",
        )

OSError: [Errno 5] Input/output error